# Entraînement d'un modèle YOLOv8n
Ce notebook a pour rôle de préparer l'environnement Ultralytics et PyTorch, puis d'entraîner, évaluer, tester et exporter au format ONNX un modèle YOLOv8n dédié à la détection des composants de faisceaux de câbles à partir du jeu de données Harness comportement detection (v1)

## 1. Installation de la bibliothèque Ultralytics

In [12]:
%pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.


## 2. Vérification de l'environnement (GPU / PyTorch)

In [2]:
import torch
import ultralytics

ultralytics.checks()
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nom du GPU : {torch.cuda.get_device_name(0)}")

Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
Setup complete  (8 CPUs, 15.8 GB RAM, 16.1/42.3 GB disk)
GPU disponible : False


## 3. Entraînement du modèle YOLOv8n

In [ ]:
from ultralytics import YOLO

# Charger le modèle pré-entraîné YOLOv8 nano
model = YOLO("yolov8n.pt")

In [ ]:
from pathlib import Path
MODELS_DIR = Path(r"D:\S5/Projet_d'expertise/projet/models").resolve()

In [ ]:
# Lancer l'entraînement
results = model.train(
    data="../data/processed/Harness comportement detection.v1i.yolov8/data.yaml",  # Chemin vers votre fichier data.yaml
    epochs=20,                     # Nombre d'époques
    imgsz=640,                     # Taille des images (640x640 par défaut)
    batch=16,                      # Taille du batch (ajustez selon la mémoire GPU)
    name="Harness_Detection",         # Nom du dossier de sauvegarde dans model
    patience=10                    # Arrêt anticipé (early stopping) si aucune amélioration
)

## 4. Évaluation du modèle sur l'ensemble de validation

In [ ]:
# Charger les meilleurs poids obtenus durant l'entraînement
best_model = YOLO("../models/Harness_Detection/weights/best.pt")

# Évaluer les métriques (mAP50, mAP50-95, précision, rappel)
metrics = best_model.val()

Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 47.513.1 MB/s, size: 27.5 KB)
val: Scanning D:\S5\Projet_d'expertise\projet\data\processed\Harness comportement detection.v1i.yolov8\valid\labels.cache... 65 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 65/65  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 705, len(boxes) = 707. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.0s/it 10.1s.3ss
                   all         65        707      0.797      0.755      0.775      0.577
         AJ20_D6_Cable         10         13      0.822      0.923      0.80

## 5. Tester le modèle (Inférence / Prédiction)

In [11]:
# Effectuer une prédiction sur une image ou un dossier de test
predictions = best_model.predict(
    source="../data/processed/Harness comportement detection.v1i.yolov8/test/images/Cable_MHEV_NC11_16_jpg.rf.704da83d6fe8286849bbfb10aae061de.jpg",  # Image unique, dossier ou vidéo
    conf=0.25,                         # Seuil de confiance minimum
    save=True                          # Sauvegarder les images avec les détections
)


image 1/1 d:\S5\Projet_d'expertise\projet\src\..\data\processed\Harness comportement detection.v1i.yolov8\test\images\Cable_MHEV_NC11_16_jpg.rf.704da83d6fe8286849bbfb10aae061de.jpg: 640x640 8 Clips, 7 Connectors, 1 Gape, 1 Info, 3 MHEV_N11_Cables, 119.3ms
Speed: 5.3ms preprocess, 119.3ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to D:\S5\Projet_d'expertise\projet\src\runs\detect\predict


## 6. Exporter le modèle (Optionnel)

In [13]:
%pip install onnx onnxslim onnxruntime

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Exporter au format ONNX (ou TensorRT, TFLite, etc.)
best_model.export(format="onnx")

Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)

PyTorch: starting from 'runs\detect\yolov8n_custom\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 13, 8400) (6.0 MB)

ONNX: starting export with onnx 1.23.2 opset 20...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success  8.4s, saved as 'runs\detect\yolov8n_custom\weights\best.onnx' (11.7 MB)

Export complete (9.0s)
Results saved to D:\S5\Projet_d'expertise\projet\src\runs\detect\yolov8n_custom\weights
Predict:         yolo predict task=detect model=runs\detect\yolov8n_custom\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs\detect\yolov8n_custom\weights\best.onnx imgsz=640 data=../data/processed/Harness comportement detection.v1i.yolov8/data.yaml  
Visualize:       https://netron.app


'runs\\detect\\yolov8n_custom\\weights\\best.onnx'